In [0]:
# Pergunta 1 — Receita total (em R$) de todos os filmes da base
display(spark.sql("""
    SELECT SUM(receita_brl) AS receita_total_brl
    FROM gold.fact_movies_performance
"""))

receita_total_brl
827319293609.35


In [0]:
# Pergunta 2 — 5 filmes com maior popularidade
display(spark.sql("""
    SELECT m.titulo, f.popularidade
    FROM gold.fact_movies_performance f
    JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
wwe survivor series 2018,2018.0


In [0]:
# Pergunta 3 — Quantidade de filmes por gênero (do maior para o menor)
display(spark.sql("""
    SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
    FROM gold.bridge_movie_genre b
    JOIN gold.dim_genres g ON g.sk_genre_id = b.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC
"""))

nome_genero,qtd_filmes
Drama,32356
Documentary,19100
Comedy,18680
Thriller,10291
Horror,9753
Romance,7657
Action,6070
Crime,4757
Animation,4488
TV Movie,4088


In [0]:
# Pergunta 4 — Top 10 filmes por receita, com RANK()
display(spark.sql("""
    SELECT titulo, receita_usd, receita_brl, posicao
    FROM (
        SELECT m.titulo, f.receita_usd, f.receita_brl,
               RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao
        FROM gold.fact_movies_performance f
        JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    )
    WHERE posicao <= 10
    ORDER BY posicao, titulo
"""))

titulo,receita_usd,receita_brl,posicao
Avengers: Endgame,2800000000.00,14311080000.00,1
Avatar: The Way of Water,2320250281.00,11859031211.22,2
AVENGERS: INFINITY WAR,2052415039.00,10490098505.83,3
spider-man: no way home,1921847111.00,9822752769.03,4
The Lion King,1663075401.00,8500144682.05,5
Top Gun: Maverick,1488732821.00,7609062321.41,6
Barbie,1428545028.00,7301436492.61,7
The Super Mario Bros. Movie,1355725263.00,6929247391.72,8
Black Panther,1349926083.00,6899607202.82,9
Star Wars: The Last Jedi,1332698830.00,6811556990.01,10


In [0]:
# Pergunta 5 — Ator com mais participações em filmes lançados nos últimos 2 anos
# (RANK = 1 mostra todos os empatados na primeira posição)
display(spark.sql("""
    WITH lim AS (
        SELECT MAX(data_lancamento) AS dt_max
        FROM gold.dim_movies
        WHERE status_filme = 'Lançado' AND data_lancamento <= current_date()
    ),
    base AS (
        SELECT p.nome_pessoa, COUNT(DISTINCT m.sk_movie_id) AS qtd_participacoes
        FROM gold.dim_movies m
        CROSS JOIN lim
        JOIN gold.bridge_movie_person b ON b.sk_movie_id = m.sk_movie_id
        JOIN gold.dim_people p ON p.sk_person_id = b.sk_person_id
        WHERE p.tipo_pessoa = 'Ator'
          AND m.status_filme = 'Lançado'
          AND m.data_lancamento >  add_months(lim.dt_max, -24)
          AND m.data_lancamento <= lim.dt_max
        GROUP BY p.nome_pessoa
    )
    SELECT nome_pessoa, qtd_participacoes, posicao
    FROM (
        SELECT nome_pessoa, qtd_participacoes,
               RANK() OVER (ORDER BY qtd_participacoes DESC) AS posicao
        FROM base
    )
    WHERE posicao = 1
"""))

nome_pessoa,qtd_participacoes,posicao
Kevin Hart,64,1


In [0]:
# Pergunta 6 — Produtora com maior lucro nos últimos 5 anos (lucro em US$ e R$; ranking por US$)
display(spark.sql("""
    WITH lim AS (
        SELECT MAX(data_lancamento) AS dt_max
        FROM gold.dim_movies
        WHERE status_filme = 'Lançado' AND data_lancamento <= current_date()
    ),
    base AS (
        SELECT c.nome_produtora,
               SUM(f.lucro_usd) AS lucro_total_usd,
               SUM(f.lucro_brl) AS lucro_total_brl
        FROM gold.dim_movies m
        CROSS JOIN lim
        JOIN gold.fact_movies_performance f ON f.sk_movie_id = m.sk_movie_id
        JOIN gold.bridge_movie_company b ON b.sk_movie_id = m.sk_movie_id
        JOIN gold.dim_companies c ON c.sk_company_id = b.sk_company_id
        WHERE m.status_filme = 'Lançado'
          AND m.data_lancamento >  add_months(lim.dt_max, -60)
          AND m.data_lancamento <= lim.dt_max
          AND f.lucro_usd IS NOT NULL
        GROUP BY c.nome_produtora
    )
    SELECT nome_produtora, lucro_total_usd, lucro_total_brl, posicao
    FROM (
        SELECT nome_produtora, lucro_total_usd, lucro_total_brl,
               RANK() OVER (ORDER BY lucro_total_usd DESC) AS posicao
        FROM base
    )
    WHERE posicao = 1
"""))

nome_produtora,lucro_total_usd,lucro_total_brl,posicao
Universal Pictures,5430017131.00,27753360558.27,1
